In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [19]:
os.getcwd()

'/Users/nikoegorov/cancer-heterogeneity/scripts/python/distance processing'

# I. Take a look into the annotation file

Sources:

1. https://www.gencodegenes.org/pages/data_format.html

In [20]:
gtf_path = "/Users/nikoegorov/cancer-heterogeneity/input/gtf/genes.gtf"
gtf_df = pd.read_csv(
    gtf_path, 
    sep="\t", 
    comment="#", 
    header=None, 
    names=["chr_name", "annotation source", "feature type", "genomic start", "genomic end", "score", "genomic strand", "genomic phase", "additional"]
)

gtf_df.head()

,chr_name,annotation source,feature type,genomic start,genomic end,score,genomic strand,genomic phase,additional
0,chr1,HAVANA,gene,11869,14409,.,+,.,"gene_id ""ENSG00000223972.5""; gene_type ""transc..."
1,chr1,HAVANA,transcript,11869,14409,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
2,chr1,HAVANA,exon,11869,12227,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
3,chr1,HAVANA,exon,12613,12721,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
4,chr1,HAVANA,exon,13221,14409,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."


# II. Extract exons for analysis

Idea: calculate distance to a variant by summing exons. 

In [21]:
exons_df = gtf_df[gtf_df["feature type"]=="exon"]
exons_df.head()

,chr_name,annotation source,feature type,genomic start,genomic end,score,genomic strand,genomic phase,additional
2,chr1,HAVANA,exon,11869,12227,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
3,chr1,HAVANA,exon,12613,12721,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
4,chr1,HAVANA,exon,13221,14409,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
6,chr1,HAVANA,exon,12010,12057,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
7,chr1,HAVANA,exon,12179,12227,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."


In [22]:
# Drop unnecessary columns
exons_df = exons_df.drop(["annotation source", "feature type", "score", "genomic phase"], axis=1)
exons_df.head()


,chr_name,genomic start,genomic end,genomic strand,additional
2,chr1,11869,12227,+,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
3,chr1,12613,12721,+,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
4,chr1,13221,14409,+,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
6,chr1,12010,12057,+,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."
7,chr1,12179,12227,+,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."


In [23]:
# Expand 9th column into separate columns
additionals = [
    "gene_id", 
    "transcript_id",
    "gene_type",
    "gene_name",
    "transcript_type",
    "transcript_name",
    "exon_number",
    "exon_id",
    "level",
    "tag"
]
# Each row consists of different additional features
# So use regex to extract the values in additional column and create a separate column for each value
for additional in additionals:
    # "?([^";]+)"?: make opening and closing quotes optional and capture the value between them
    exons_df[additional] = exons_df["additional"].str.extract(f'{additional} "?([^";]+)"?')
exons_df = exons_df.drop("additional", axis=1)
print(exons_df.head())

  chr_name  genomic start  genomic end genomic strand            gene_id  \
2     chr1          11869        12227              +  ENSG00000223972.5   
3     chr1          12613        12721              +  ENSG00000223972.5   
4     chr1          13221        14409              +  ENSG00000223972.5   
6     chr1          12010        12057              +  ENSG00000223972.5   
7     chr1          12179        12227              +  ENSG00000223972.5   

       transcript_id                           gene_type gene_name  \
2  ENST00000456328.2  transcribed_unprocessed_pseudogene   DDX11L1   
3  ENST00000456328.2  transcribed_unprocessed_pseudogene   DDX11L1   
4  ENST00000456328.2  transcribed_unprocessed_pseudogene   DDX11L1   
6  ENST00000450305.2  transcribed_unprocessed_pseudogene   DDX11L1   
7  ENST00000450305.2  transcribed_unprocessed_pseudogene   DDX11L1   

                      transcript_type transcript_name exon_number  \
2                              lncRNA     DDX11L1-202

In [24]:
basic_transcripts = exons_df[exons_df["tag"]=="basic"]
# plot the distribution of basic and non-basic transcripts


In [25]:
# Save the processed df to a CSV file
output_dir = "/Users/nikoegorov/cancer-heterogeneity/output/gtf"
os.makedirs(output_dir, exist_ok=True)
basic_transcripts.to_csv(f"{output_dir}/gtf_exons_basic.csv", index=False)